<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Exorust/TorchLeet/blob/main/llm/Byte-Pair-Encoder/BPE-q3.ipynb)

Check your work: `!pip install torchleet` then `from torchleet import check; check("byte-pair-encoder", ...)`


## Problem: Write a Byte Pain Encoder in Python

### Problem Statement
Implement a **Transformer model** in PyTorch by completing the required sections. The model should consist of an embedding layer, a Transformer encoder, and an output layer for sequence processing and prediction.

### Requirements
1. **Define the Transformer Model Architecture**:
   - **Embedding Layer**:
     - Implement a layer to transform input data into a higher-dimensional space.
     - Use a `torch.nn.Linear` or `torch.nn.Embedding` layer to create embeddings from the input.
   - **Transformer Encoder**:
     - Use `torch.nn.TransformerEncoder` or `torch.nn.Transformer` to process sequences with attention.
     - Configure parameters such as the number of attention heads and encoder layers.
   - **Output Layer**:
     - Add a fully connected (linear) layer to reduce the transformer's sequence output into the desired output dimension.

2. **Implement the Forward Method**:
   - Map the input to the higher-dimensional space using the embedding layer.
   - Pass the transformed input through the Transformer encoder.
   - Use the output layer to convert the encoded sequence into predictions.

### Constraints
- Handle input padding correctly for variable-length sequences.
- Ensure compatibility with batch processing by correctly shaping input and output tensors.


In [41]:
from collections import defaultdict, Counter

def get_vocab(corpus):
    """Creates a vocabulary with words split into characters and a special end-of-word token."""
    vocab = []
    for word in corpus:
      word_list = list(word)
      word_list.append("</w>")
      vocab.append(tuple(word_list))

    return Counter(vocab)

def get_stats(vocab):
    """Counts frequency of adjacent symbol pairs."""

    # we have an input which is word<e> -> count
    # we want to make a frequncy of which 2 elements are next to each other
    pairs = []
    for key, value in vocab.items():
      for i in range(1, len(key)):
        keySlice = key[i - 1:i + 1]
        for _ in range(value):
          pairs.append(keySlice)

    return Counter(pairs)



def merge_vocab(pair, vocab):
    """Merges the most frequent pair into a single symbol."""
    newVocab = vocab.copy()
    for word in vocab.keys():
      for i in range(1, len(word)):
        if word[i-1:i+1] == pair:
          newWord = word[:i-1] + (pair[0] + pair[1], ) + word[i+1:]
          newVocab[newWord] = newVocab.pop(word)

    return newVocab


def byte_pair_encoding(corpus, num_merges=10):
    """Performs BPE on a corpus."""
    vocab = get_vocab(corpus)

    ## vocab looks like
    """
    Counter({'low': 1, 'lowest': 1, 'newer': 1, 'wider': 1})
    """
    merges = []

    for i in range(num_merges):
      pairs = get_stats(vocab)
      # ensure that this is deterministic to first element
      sorted_pairs = sorted(pairs.items(), key=lambda x: x[1], reverse=True)
      vocab = merge_vocab(sorted_pairs[0][0], vocab)
      merges.append(sorted_pairs[0][0])

    return vocab, merges

# Example usage
corpus = ["low", "lowest", "newer", "wider"]
final_vocab, merge_operations = byte_pair_encoding(corpus, num_merges=10)
print("\nFinal Vocabulary:")
print(final_vocab)



Final Vocabulary:
Counter({('ne', 'we', 'r</w>'): 1, ('low</w>',): 1, ('lowes', 't', '</w>'): 1, ('wid', 'e', 'r</w>'): 1})


In [46]:
def test_get_vocab():
    corpus = ["test"]
    vocab = get_vocab(corpus)
    assert vocab == {('t', 'e', 's', 't', '</w>'): 1}
    print("✓ test_get_vocab passed")

def test_get_stats():
    vocab = {('t', 'e', 's', 't', '</w>'): 1}
    stats = get_stats(vocab)
    expected = {
        ('t', 'e'): 1,
        ('e', 's'): 1,
        ('s', 't'): 1,
        ('t', '</w>'): 1
    }
    assert stats == expected
    print("✓ test_get_stats passed")

def test_merge_vocab():
    vocab = {('t', 'e', 's', 't', '</w>'): 1}
    merged = merge_vocab(('e', 's'), vocab)
    expected = {('t', 'es', 't', '</w>'): 1}
    assert merged == expected
    print("✓ test_merge_vocab passed")

def test_bpe_sequence():
    corpus = ["low", "lower", "newest", "widest"]
    final_vocab, merges = byte_pair_encoding(corpus, num_merges=5)
    assert isinstance(final_vocab, dict)
    assert all(isinstance(pair, tuple) for pair in merges)
    assert len(merges) == 5
    print("✓ test_bpe_sequence passed")

# Run all tests
test_get_vocab()
test_get_stats()
test_merge_vocab()
test_bpe_sequence()


✓ test_get_vocab passed
✓ test_get_stats passed
✓ test_merge_vocab passed
✓ test_bpe_sequence passed
